# 🧬 CHITIN
### Causal Heuristics Isolating Transcriptomic Intervention Noise

**Objective:** Strip systematic pan-perturbation variation from metacell expression matrices via localized manifold subtraction, producing a pure Δ-matrix of causal perturbation signatures for downstream GRN inference.

**Input:** SPORE Phase 8 metacell `.h5ad` files (train/val/test)  
**Output:** Δ-transformed `.h5ad` files where each perturbed metacell's expression = $x_i - \vec{N}_i$ (localized control baseline)  
**Downstream:** GuanLab PSGRN (LightGBM) → FUNGI → SPECTRA  

---

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import scanpy as sc
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')

from src.utils import load_config, setup_logger, apply_chitin_style
from src.utils import log_memory, snapshot, force_gc
from src import engine
from src import diagnostics as diag
from src import plotting as cplt

cfg = load_config('chitin_config.yaml')
logger = setup_logger(cfg)
apply_chitin_style(cfg)

print(f"CHITIN initialized")
print(f"  Input dir : {cfg['paths']['_input']}")
print(f"  k         : {cfg['knn']['k']}")
print(f"  n_PCs     : {cfg['knn']['n_pcs']}")
log_memory(logger, 'startup')

## Load SPORE Metacell Data

Loading the Phase 8 metacell-aggregated train split.  
Val and test splits will be transformed using the **same PCA space** derived from train.

In [ ]:
# Load training metacells
train_path = cfg['paths']['_input'] / cfg['paths']['train_file']
adata_train = sc.read_h5ad(train_path)
snapshot(adata_train, 'Train metacells loaded', logger)

# Quick inspection
print(f"\n.obs columns: {list(adata_train.obs.columns)}")
print(f"Perturbation column ('{cfg['dataset']['perturbation_col']}'):")
print(adata_train.obs[cfg['dataset']['perturbation_col']].value_counts().head(10))

---
## Pre-CHITIN Diagnostics

*Quantify the systematic variation BEFORE applying localized subtraction.  
This establishes the baseline that CHITIN needs to eliminate.*

### Dynamic Latent Space (Fresh PCA on Metacells)

We compute a **fresh PCA** directly on the metacell data — NOT inherited from SPORE.  
The Phase 8 aggregation fundamentally altered the topological geometry.

In [ ]:
# Instantiate the CHITIN model and fit the training reference manifold
# This automatically computes the fresh PCA and fits the KNN on control metacells
chitin_model = engine.ChitinModel()
adata_train = chitin_model.fit(adata_train, cfg, logger)

In [ ]:
# PCA variance explained
cplt.plot_pca_variance(adata_train, cfg)

In [ ]:
# Latent space visualization (PC1 vs PC2)
cplt.plot_latent_space(adata_train, cfg)

### Systema Centroid Analysis (Pre-CHITIN)

Compute $C_{ctrl}$, $O_{pert}$, and the systematic variation vector $\vec{V}$.  
Measure cosine similarity between perturbation-specific shifts and $\vec{V}$.  

⚠️ These centroids are for **evaluation only** — they are NOT used in the CHITIN transformation.

In [ ]:
# Compute Systema centroids on pre-CHITIN data
C_ctrl_pre, O_pert_pre, V_pre = diag.compute_systema_centroids(
    adata_train, cfg, logger)

# Compute per-perturbation cosine similarities
cos_ctrl_pre, cos_pert_pre, pert_names = diag.compute_perturbation_cosines(
    adata_train, C_ctrl_pre, O_pert_pre, cfg, logger)

In [ ]:
# Cosine distributions (pre-CHITIN)
cplt.plot_cosine_distributions(cos_ctrl_pre, cos_pert_pre, cfg, label='pre')

### k-Sensitivity Analysis

Sweep across k values to verify that the chosen k produces **localized** neighborhoods  
rather than collapsing toward the global control mean.

- **High basal variance** → neighborhoods are diverse (good, localized)
- **Low basal variance** → neighborhoods converge to global mean (bad, rank-order invariance returns)

In [ ]:
if cfg['diagnostics']['k_sweep']:
    k_vals, basal_vars, mean_dists = diag.k_sensitivity_sweep(
        adata_train, cfg, logger)
    cplt.plot_k_sensitivity(k_vals, basal_vars, mean_dists,
                            chosen_k=cfg['knn']['k'], cfg=cfg)

---
## CHITIN Transformation (Train Split)

**The core operation:** For each perturbed metacell $x_i$:
1. Find its $k$ nearest **control** neighbors in the fresh PCA latent space
2. Compute the localized basal vector $\vec{N}_i$ = mean of those neighbors
3. Extract the causal delta: $\Delta X_i = x_i - \vec{N}_i$

Control metacells are set to $\Delta = 0$ (they ARE the baseline).

In [ ]:
# Extract the causal delta for the training split using the fitted model
adata_train_delta = chitin_model.transform(adata_train, cfg, logger, label="Train")

---
## Post-CHITIN Diagnostics

*Verify that systematic variation has been reduced.*

In [ ]:
# ── Rank-Order Disruption (the metric that actually matters for LightGBM) ──
mean_rho, rank_corrs, sampled_genes = diag.compute_rank_disruption(
    adata_train, adata_train_delta, cfg, logger)
cplt.plot_rank_disruption(rank_corrs, sampled_genes, cfg)

In [ ]:
# ── Pairwise Perturbation Discrimination ──
dist_pre, dist_post = diag.compute_pairwise_discrimination(
    adata_train, adata_train_delta, cfg, logger)
cplt.plot_pairwise_discrimination(dist_pre, dist_post, cfg)

### Delta Magnitude Analysis

*Which perturbations have the strongest causal signals after CHITIN?*

In [ ]:
# Delta magnitude statistics
delta_df, delta_norms = diag.compute_delta_magnitudes(
    adata_train_delta, cfg, logger)

In [ ]:
# Delta magnitude distribution
cplt.plot_delta_magnitude_distribution(delta_norms, cfg)

In [ ]:
# Top perturbations by causal signal strength
cplt.plot_top_perturbation_deltas(delta_df, cfg, n_top=30)

---
## Transform Val & Test Splits

Apply CHITIN independently to val and test splits.  
Each split gets its own fresh PCA and KNN — no leakage.

In [ ]:
# Val split
val_path = cfg['paths']['_input'] / cfg['paths']['val_file']
adata_val = sc.read_h5ad(val_path)
snapshot(adata_val, 'Val metacells loaded', logger)

# Project and transform using the TRAINING reference manifold
adata_val_delta = chitin_model.transform(adata_val, cfg, logger, label="Val")
del adata_val
force_gc(logger)

In [ ]:
# Test split
test_path = cfg['paths']['_input'] / cfg['paths']['test_file']
adata_test = sc.read_h5ad(test_path)
snapshot(adata_test, 'Test metacells loaded', logger)

# Project and transform using the TRAINING reference manifold
adata_test_delta = chitin_model.transform(adata_test, cfg, logger, label="Test")
del adata_test
force_gc(logger)

---
## Save CHITIN Outputs

Δ-transformed `.h5ad` files ready for GuanLab PSGRN.

In [ ]:
output_dir = cfg['paths']['_output']
suffix = cfg['output']['suffix']
dataset = cfg['dataset']['name']

splits = {'train': adata_train_delta, 'val': adata_val_delta, 'test': adata_test_delta}

for key, ad in splits.items():
    fname = f"{dataset}_{key}{suffix}.h5ad"
    path = output_dir / fname
    ad.write_h5ad(path)
    logger.info(f"Saved {key} → {path}")
    print(f"  ✓ {key}: {ad.n_obs:,} metacells × {ad.n_vars:,} genes → {fname}")

print(f"\nAll CHITIN outputs saved to: {output_dir}")
log_memory(logger, 'pipeline complete')

---
## Summary

| Metric | Pre-CHITIN | Post-CHITIN |
|--------|------------|-------------|
| $\vec{V}$ (systematic variation) | computed above | computed above |
| Mean cosine (ctrl ref) | computed above | computed above |
| Mean cosine (pert ref) | computed above | computed above |

In [ ]:
print("\n" + "═" * 60)
print("  CHITIN Pipeline Complete")
print("═" * 60)
print(f"  Rank-order disruption (mean Spearman ρ): {mean_rho:.4f}")
print(f"  Pairwise discrimination change: "
      f"{(dist_post.mean()-dist_pre.mean())/dist_pre.mean()*100:+.1f}%")
print(f"  Mean cosine (ctrl ref):")
print(f"    Pre-CHITIN:  {cos_ctrl_pre.mean():.4f}")
print(f"    Post-CHITIN: {cos_ctrl_post.mean():.4f}")
print(f"  k = {cfg['knn']['k']} neighbors")
print(f"  {cfg['knn']['n_pcs']} PCA components")
print(f"  Train: {adata_train_delta.n_obs:,} metacells")
print(f"  Val:   {adata_val_delta.n_obs:,} metacells")
print(f"  Test:  {adata_test_delta.n_obs:,} metacells")
print("═" * 60)
log_memory(logger, 'final')